In [2]:
import time
from sklearn.metrics import accuracy_score

from product_quantization import ProductQuantizationKNN

# Evaluate on MNIST data set

### Read the data

Borrowed from; https://www.cntk.ai/pythondocs/CNTK_103A_MNIST_DataLoader.html

In [3]:
import os
import gzip
import struct
import numpy as np

try:
    from urllib.request import urlretrieve
except ImportError:
    from urllib import urlretrieve

In [4]:
# 用于加载 MNIST 图像并将其拆分为训练集和测试集的函数。
# - loadData 函数读取图像数据并将其格式化为 28x28 的长数组。
# - loadLabels 函数读取每张图像对应的标签数据。
# - try_download 函数将下载的图像和标签数据打包，供 CNTK 文本阅读器读取。

import os
import gzip
import struct
import numpy as np
from urllib.request import urlretrieve

def loadData(src, cimg):
    """
    从指定的源 URL 下载图像数据并格式化为 28x28 长数组。

    :param src: 图像数据的 URL 地址。
    :param cimg: 图像的数量，用于验证下载数据的正确性。
    :return: 格式化后的图像数据，形状为 (cimg, 784)。
    """
    print('Downloading ' + src)
    gzfname, h = urlretrieve(src, './delete.me')  # 下载并保存为临时文件
    print('Done.')
    try:
        with gzip.open(gzfname) as gz:
            # 读取文件头部的魔术数
            n = struct.unpack('I', gz.read(4))
            if n[0] != 0x3080000:
                raise Exception('Invalid file: unexpected magic number.')
            # 读取图像的数量并进行验证
            n = struct.unpack('>I', gz.read(4))[0]
            if n != cimg:
                raise Exception('Invalid file: expected {0} entries.'.format(cimg))
            # 读取图像的行数和列数，并验证是否为 28x28
            crow = struct.unpack('>I', gz.read(4))[0]
            ccol = struct.unpack('>I', gz.read(4))[0]
            if crow != 28 or ccol != 28:
                raise Exception('Invalid file: expected 28 rows/cols per image.')
            # 读取图像数据，并将其转换为 uint8 类型的 NumPy 数组
            res = np.fromstring(gz.read(cimg * crow * ccol), dtype=np.uint8)
    finally:
        os.remove(gzfname)  # 删除临时文件
    return res.reshape((cimg, crow * ccol))  # 返回格式化后的图像数据

def loadLabels(src, cimg):
    """
    从指定的源 URL 下载标签数据。

    :param src: 标签数据的 URL 地址。
    :param cimg: 标签的数量，用于验证下载数据的正确性。
    :return: 标签数据，形状为 (cimg, 1)。
    """
    print('Downloading ' + src)
    gzfname, h = urlretrieve(src, './delete.me')  # 下载并保存为临时文件
    print('Done.')
    try:
        with gzip.open(gzfname) as gz:
            # 读取文件头部的魔术数
            n = struct.unpack('I', gz.read(4))
            if n[0] != 0x1080000:
                raise Exception('Invalid file: unexpected magic number.')
            # 读取标签的数量并进行验证
            n = struct.unpack('>I', gz.read(4))
            if n[0] != cimg:
                raise Exception('Invalid file: expected {0} rows.'.format(cimg))
            # 读取标签数据，并将其转换为 uint8 类型的 NumPy 数组
            res = np.fromstring(gz.read(cimg), dtype=np.uint8)
    finally:
        os.remove(gzfname)  # 删除临时文件
    return res.reshape((cimg, 1))  # 返回格式化后的标签数据

def try_download(dataSrc, labelsSrc, cimg):
    """
    下载图像和标签数据，并将它们组合在一起。

    :param dataSrc: 图像数据的 URL 地址。
    :param labelsSrc: 标签数据的 URL 地址。
    :param cimg: 图像和标签的数量，用于验证数据的正确性。
    :return: 图像数据和标签数据组合在一起的数组。
    """
    data = loadData(dataSrc, cimg)  # 下载并加载图像数据
    labels = loadLabels(labelsSrc, cimg)  # 下载并加载标签数据
    return np.hstack((data, labels))  # 将图像数据和标签数据水平拼接

In [5]:
# 训练图像和标签数据的 URL
url_train_image = 'https://ossci-datasets.s3.amazonaws.com/mnist/train-images-idx3-ubyte.gz'
url_train_labels = 'https://ossci-datasets.s3.amazonaws.com/mnist/train-labels-idx1-ubyte.gz'
num_train_samples = 60000  # 训练样本的数量

print("Downloading train data")
train = try_download(url_train_image, url_train_labels, num_train_samples)
# 使用 `try_download` 函数从指定的 URL 下载训练数据，包括图像和标签数据，并存储为 `train` 变量

# 测试图像和标签数据的 URL
url_test_image = 'https://ossci-datasets.s3.amazonaws.com/mnist/t10k-images-idx3-ubyte.gz'
url_test_labels = 'https://ossci-datasets.s3.amazonaws.com/mnist/t10k-labels-idx1-ubyte.gz'
num_test_samples = 10000  # 测试样本的数量

print("Downloading test data")
test = try_download(url_test_image, url_test_labels, num_test_samples)
# 使用 `try_download` 函数从指定的 URL 下载测试数据，包括图像和标签数据，并存储为 `test` 变量


Done.


C:\Users\19108\AppData\Local\Temp\ipykernel_33728\3398142394.py:39: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  res = np.fromstring(gz.read(cimg * crow * ccol), dtype=np.uint8)


Done.


C:\Users\19108\AppData\Local\Temp\ipykernel_33728\3398142394.py:66: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  res = np.fromstring(gz.read(cimg), dtype=np.uint8)


Done.
Done.


In [6]:
# 从训练和测试数据中提取标签和特征

train_labels = train[:, -1]  # 提取训练数据中的标签（最后一列）
train_data = train[:, :-1]   # 提取训练数据中的特征（除最后一列之外的所有列）
test_labels = test[:, -1]    # 提取测试数据中的标签（最后一列）
test_data = test[:, :-1]     # 提取测试数据中的特征（除最后一列之外的所有列）

# 输出特征数据的形状，以确认数据加载和拆分是否正确
print('Train features shape:', train_data.shape)
print('Test features shape:', test_data.shape)


Train features shape: (60000, 784)
Test features shape: (10000, 784)


### The hyper parameters

In [7]:
# Number of nearest-neighbors
k = 10

## Evaluate PQKNN approach

### Multi core

In [8]:
# 创建一个 ProductQuantizationKNN (PQKNN) 分类器的实例
pqknn = ProductQuantizationKNN(7, 4)
# 这里，参数 7 表示将特征向量分成 7 个子向量，参数 4 表示每个子向量使用 2^4 = 16 个聚类（KMeans）

# 记录压缩开始的时间
start = time.time()

# 使用 PQKNN 分类器对训练数据进行压缩
pqknn.compress(train_data, train_labels)
# `compress` 方法将训练数据分成多个子向量，并使用 KMeans 聚类对其进行压缩，以便后续快速进行近邻搜索

# 记录压缩结束的时间
end = time.time()

# 输出压缩训练数据所花费的时间
print('Compressing the train_data to PQKNN classifier took ' + str(end - start) + ' seconds.')


Compressing the train_data to PQKNN classifier took 2.1129348278045654 seconds.


Since $c$=4, we use np.uint8 (thus 1 byte) to store the centroid_ids in the compressedData array

In [8]:
# 输出压缩后的数据形状
print('Compressed data shape:', pqknn.compressed_data.shape)
# `pqknn.compressed_data.shape` 显示压缩后的训练数据的形状，用于验证压缩操作是否正确执行

# 输出压缩后的数据大小（以字节为单位）
print('Compressed data in bytes:', pqknn.compressed_data.nbytes)
# `pqknn.compressed_data.nbytes` 计算压缩后的训练数据占用的内存大小，单位是字节

# 输出原始训练数据大小（以字节为单位）
print('Original data in bytes:', train_data.nbytes)
# `train_data.nbytes` 计算未压缩的原始训练数据占用的内存大小，单位是字节

# 计算并输出压缩比
print('Compression factor:', train_data.nbytes / pqknn.compressed_data.nbytes)
# 计算压缩因子（原始数据大小除以压缩数据大小）
# 压缩比表示通过产品量化方法压缩数据所减少的内存占用量


In [9]:
# 记录预测开始的时间
start = time.time()

# 使用 PQKNN 分类器对测试数据进行预测
preds = pqknn.predict(test_data, k)
# `pqknn.predict` 方法对测试数据 `test_data` 进行预测，`k` 是 k 近邻中考虑的邻居数
# `preds` 变量存储预测的结果

# 记录预测结束的时间
end = time.time()

# 输出使用 PQKNN 分类器对测试数据进行预测所花费的时间
print('Predicting the test_data with PQKNN classifier took ' + str(end - start) + ' seconds.')


In [10]:
# 计算并输出使用 PQKNN 分类器对测试数据的预测准确率
print('Accuracy: ' + str(accuracy_score(test_labels, preds) * 100) + '%')
# `accuracy_score` 函数计算测试数据的实际标签 `test_labels` 和预测标签 `preds` 之间的准确率
# 将准确率乘以 100 以百分比格式显示，并输出结果


#### If we increase the number of clusters ($c$), then the accuracy increases (together with the storage)

In [11]:
# 创建一个 ProductQuantizationKNN (PQKNN) 分类器的实例
pqknn = ProductQuantizationKNN(7, 9)
# 参数解释：
# - 7 表示将特征向量分成 7 个子向量，以便进行产品量化。
# - 9 表示每个子向量使用 2^9 = 512 个聚类（KMeans）。这个参数决定了每个子向量被量化为多少个类。

# 记录压缩开始的时间
start = time.time()

# 使用 PQKNN 分类器对训练数据进行压缩
pqknn.compress(train_data, train_labels)
# `compress` 方法对训练数据进行产品量化，将高维数据分成多个子向量并用 KMeans 进行聚类压缩。
# 这种压缩方式加快了后续的 k-NN 搜索操作，尤其在处理大规模高维数据时非常有用。

# 记录压缩结束的时间
end = time.time()

# 输出压缩训练数据所花费的时间
print('Compressing the train_data to PQKNN classifier took ' + str(end - start) + ' seconds.')


Since $c$=9, we use np.uint16 (thus 2 bytes) to store the centroid_ids in the compressedData array  
-> Resulting in twice the storage size of the example above

In [12]:
# 输出压缩后的数据形状
print('Compressed data shape:', pqknn.compressed_data.shape)
# `pqknn.compressed_data.shape` 显示压缩后的训练数据的形状，用于确认压缩操作是否成功以及数据结构是否正确

# 输出压缩后的数据大小（以字节为单位）
print('Compressed data in bytes:', pqknn.compressed_data.nbytes)
# `pqknn.compressed_data.nbytes` 计算压缩后的训练数据在内存中占用的大小（字节数）
# 这可以帮助评估压缩后的数据内存消耗是否显著降低

# 输出原始训练数据大小（以字节为单位）
print('Original data in bytes:', train_data.nbytes)
# `train_data.nbytes` 计算未压缩的原始训练数据在内存中占用的大小（字节数）
# 用于与压缩后的数据进行比较，评估压缩效果

# 计算并输出压缩比
print('Compression factor:', train_data.nbytes / pqknn.compressed_data.nbytes)
# 计算压缩因子，即原始数据大小除以压缩数据大小
# 压缩因子表示压缩后的数据比原始数据减少了多少内存占用。压缩因子越大，表示压缩效果越好


In [13]:
# 记录预测开始的时间
start = time.time()

# 使用 PQKNN 分类器对测试数据进行预测
preds = pqknn.predict(test_data, k)
# `pqknn.predict` 方法对测试数据 `test_data` 进行预测，`k` 是 k-NN 算法中要考虑的最近邻的数量
# `preds` 变量存储了预测结果，即测试数据的分类标签

# 记录预测结束的时间
end = time.time()

# 输出使用 PQKNN 分类器对测试数据进行预测所花费的时间
print('Predicting the test_data with PQKNN classifier took ' + str(end - start) + ' seconds.')


In [14]:
print('Accuracy: ' + str(accuracy_score(test_labels, preds)*100) + '%')

#### With some significantly smaller space we obtain a (good) accuracy between the first and the second example. With a very good compression factor and fast compression and predict time!

In [15]:
# 创建一个 ProductQuantizationKNN (PQKNN) 分类器的实例
pqknn = ProductQuantizationKNN(4, 8)
# 参数解释：
# - 4 表示将特征向量分成 4 个子向量进行产品量化，这样可以在保持数据表示的基础上提高搜索效率。
# - 8 表示每个子向量使用 2^8 = 256 个聚类（KMeans）。这个参数指定每个子向量被量化为 256 个类别。

# 记录压缩开始的时间
start = time.time()

# 使用 PQKNN 分类器对训练数据进行压缩
pqknn.compress(train_data, train_labels)
# `compress` 方法将训练数据分成多个子向量，并使用 KMeans 聚类对其进行压缩。
# 这种产品量化方法减少了数据维度并加速了后续的 k-NN 搜索操作。

# 记录压缩结束的时间
end = time.time()

# 输出压缩训练数据所花费的时间
print('Compressing the train_data to PQKNN classifier took ' + str(end - start) + ' seconds.')


Since $c$=8, we use np.uint8 (thus 1 byte) to store the centroid_ids in the compressedData array

In [16]:
print('Compressed data shape:', pqknn.compressed_data.shape)
print('Compressed data in bytes:', pqknn.compressed_data.nbytes)
print('Original data in bytes:', train_data.nbytes)
print('Compression factor:', train_data.nbytes / pqknn.compressed_data.nbytes)

In [17]:
start = time.time()
preds = pqknn.predict(test_data, k)
end = time.time()
print('Predicting the test_data with PQKNN classifier took ' + str(end - start) + ' seconds.')

In [18]:
print('Accuracy: ' + str(accuracy_score(test_labels, preds)*100) + '%')

## Evaluate SKlearn K-NN approach on data

### Single core

In [19]:
from sklearn.neighbors import KNeighborsClassifier

# 创建一个 KNeighborsClassifier 实例
kNN = KNeighborsClassifier(n_neighbors=k)  # `k` 是 k-NN 算法中要考虑的最近邻数量
# 参数解释：
# - `n_neighbors=k`：指定要考虑的邻居数，这将影响分类决策的准确性和效果
# - `n_jobs=1`（默认值）：表示使用单个线程进行计算

# 记录模型拟合开始的时间
start = time.time()

# 使用 k-NN 分类器拟合训练数据
kNN.fit(train_data, train_labels)
# `fit` 方法将训练数据 `train_data` 和对应的标签 `train_labels` 传入 k-NN 模型中进行训练
# k-NN 是一种懒惰学习算法，训练过程实际上只是存储数据，以备后续的距离计算

# 记录模型拟合结束的时间
end = time.time()

# 输出拟合训练数据所花费的时间
print('Fitting the train_data to SKlearn KNN classifier took ' + str(end - start) + ' seconds.')


In [20]:
start = time.time()
preds = kNN.predict(test_data)
end = time.time()
print('Predicting the test_data with SKlearn KNN classifier took ' + str(end - start) + ' seconds.')

In [21]:
print('Accuracy: ' + str(accuracy_score(test_labels, preds)*100) + '%')

### Multi core

In [22]:
from sklearn.neighbors import KNeighborsClassifier

# 创建一个 KNeighborsClassifier 实例，并启用并行计算
kNN = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
# 参数解释：
# - `n_neighbors=k`：指定 k-NN 算法中要考虑的最近邻的数量，影响分类决策的准确性
# - `n_jobs=-1`：启用并行计算，使用所有可用的 CPU 核心来加速模型的训练和预测
#   - `-1` 表示使用计算机的所有可用 CPU 资源，从而提高计算效率

# 记录模型拟合开始的时间
start = time.time()

# 使用 k-NN 分类器拟合训练数据
kNN.fit(train_data, train_labels)
# `fit` 方法将训练数据 `train_data` 和对应的标签 `train_labels` 传入 k-NN 模型进行训练
# k-NN 是一种惰性学习算法，训练过程只是存储数据以备后续的预测使用

# 记录模型拟合结束的时间
end = time.time()

# 输出拟合训练数据所花费的时间
print('Fitting the train_data to SKlearn KNN classifier took ' + str(end - start) + ' seconds.')


In [23]:
start = time.time()
preds = kNN.predict(test_data)
end = time.time()
print('Predicting the test_data with SKlearn KNN classifier took ' + str(end - start) + ' seconds.')

In [24]:
print('Accuracy: ' + str(accuracy_score(test_labels, preds)*100) + '%')